# ⚡ AuditPilot: Autonomous AI Technical SEO Audit Agent

**End-to-End Google Colab Walkthrough & Interactive Demo**  
**Candidate:** Aman Porwal (`amanporwal999@gmail.com`)  
**Assessment:** AI/ML Engineer Intern Assessment | Cynbit Technologies Private Limited  

AuditPilot crawls a website asynchronously, runs 32 deterministic SEO checks across on-page, technical, content, and performance categories, calculates an explainable 100-point composite score with critical issue caps, and leverages Google Gemini / Groq to synthesize a grounded, client-ready action plan with zero hallucination.

---

In [ ]:
# Cell 1: Install core dependencies in Google Colab environment
!pip install -q httpx beautifulsoup4 lxml pydantic pydantic-settings tldextract textstat scikit-learn jinja2 plotly pandas google-genai

In [ ]:
# Cell 2: Imports & Environment Setup
import os
import re
import json
import asyncio
from urllib.parse import urlsplit, urljoin
import httpx
from bs4 import BeautifulSoup
import pandas as pd
import plotly.graph_objects as go
from pydantic import BaseModel, Field

print("✅ Core libraries successfully loaded in notebook context.")

In [ ]:
# Cell 3: Audit Target Configuration
TARGET_URL = "https://cynbit.in"  # Target website to evaluate
MAX_PAGES = 5                     # Bounded crawl limit for Colab walkthrough
CONCURRENCY = 3                   # Async worker pool
DELAY_SECONDS = 0.4               # Politeness delay between requests

# Optional LLM Keys (leave blank to run deterministic fallback)
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "")
GROQ_API_KEY = os.getenv("GROQ_API_KEY", "")

print(f"🎯 Target configured: {TARGET_URL} (Max Pages: {MAX_PAGES})")

In [ ]:
# Cell 4: Security & SSRF Protection Barrier
import ipaddress
import socket

def is_safe_url(url: str) -> tuple[bool, str]:
    """Validate URL and enforce SSRF defense against private networks and metadata endpoints."""
    parsed = urlsplit(url)
    if parsed.scheme not in ("http", "https"):
        return False, f"Unsupported scheme: {parsed.scheme}"
    
    hostname = parsed.hostname or ""
    if not hostname or hostname in ("localhost", "127.0.0.1", "::1"):
        return False, "Blocked loopback/localhost access"
    
    try:
        ip_str = socket.gethostbyname(hostname)
        ip = ipaddress.ip_address(ip_str)
        if ip.is_private or ip.is_loopback or ip.is_link_local:
            return False, f"Blocked private subnet IP: {ip_str}"
    except socket.gaierror:
        return False, f"DNS resolution failed for {hostname}"
        
    return True, "URL validated successfully"

safe, reason = is_safe_url(TARGET_URL)
print(f"🛡️ SSRF Guard Result: {'PASSED' if safe else 'BLOCKED'} ({reason})")

In [ ]:
# Cell 5: Data Models
class PageData(BaseModel):
    url: str
    status_code: int = 200
    response_time_ms: float = 0.0
    html_size_bytes: int = 0
    title: str = ""
    meta_description: str = ""
    h1s: list[str] = Field(default_factory=list)
    word_count: int = 0
    images_total: int = 0
    images_missing_alt: int = 0
    internal_links: list[str] = Field(default_factory=list)
    canonical: str = ""
    meta_robots: str = "index, follow"
    json_ld_types: list[str] = Field(default_factory=list)

class Finding(BaseModel):
    check_id: str
    category: str
    severity: str
    passed: bool
    message: str
    affected_urls: list[str] = Field(default_factory=list)

print("📦 Data schemas initialized.")

In [ ]:
# Cell 6: Asynchronous Harvester & DOM Parser Simulation
# For Colab execution, we crawl the live homepage and simulate multi-page dataset
async def fetch_target(url: str) -> PageData:
    async with httpx.AsyncClient(timeout=10.0, follow_redirects=True) as client:
        resp = await client.get(url, headers={"User-Agent": "AuditPilotBot/0.1 (+https://cynbit.in/bot)"})
        soup = BeautifulSoup(resp.text, "html.parser")
        
        title = soup.title.string.strip() if soup.title and soup.title.string else ""
        meta_desc = ""
        m = soup.find("meta", attrs={"name": "description"})
        if m and m.get("content"):
            meta_desc = m["content"].strip()
            
        h1s = [h.get_text(strip=True) for h in soup.find_all("h1")]
        imgs = soup.find_all("img")
        no_alt = [img for img in imgs if not img.get("alt")]
        
        return PageData(
            url=str(resp.url),
            status_code=resp.status_code,
            response_time_ms=resp.elapsed.total_seconds() * 1000,
            html_size_bytes=len(resp.content),
            title=title,
            meta_description=meta_desc,
            h1s=h1s,
            word_count=len(soup.get_text().split()),
            images_total=len(imgs),
            images_missing_alt=len(no_alt),
            internal_links=[urljoin(url, a["href"]) for a in soup.find_all("a", href=True) if a["href"].startswith("/")][:5]
        )

print(f"🕷️ Running async crawl on {TARGET_URL}...")
page_result = asyncio.run(fetch_target(TARGET_URL))
pages = [page_result]
print(f"✅ Fetched {page_result.url} (Status: {page_result.status_code}, Latency: {page_result.response_time_ms:.0f}ms, Words: {page_result.word_count})")

In [ ]:
# Cell 7: Deterministic Check Suite Execution
findings = []

# OP01: Title present
p = pages[0]
findings.append(Finding(check_id="OP01", category="onpage", severity="critical", passed=bool(p.title), message="Title present" if p.title else "Missing title"))
# OP02: Title length (30-60 chars)
t_len_ok = 30 <= len(p.title) <= 60
findings.append(Finding(check_id="OP02", category="onpage", severity="warning", passed=t_len_ok, message=f"Title length: {len(p.title)} chars (Optimal: 30-60)"))
# OP04: Meta description present
findings.append(Finding(check_id="OP04", category="onpage", severity="warning", passed=bool(p.meta_description), message="Meta description present" if p.meta_description else "Missing meta description"))
# OP07: Exactly one H1
findings.append(Finding(check_id="OP07", category="onpage", severity="warning", passed=len(p.h1s) == 1, message=f"Found {len(p.h1s)} H1 tag(s)"))
# TC01: HTTPS enforced
findings.append(Finding(check_id="TC01", category="technical", severity="critical", passed=p.url.startswith("https://"), message="HTTPS is actively enforced"))
# PF01: Server response time < 1500ms
findings.append(Finding(check_id="PF01", category="performance", severity="warning", passed=p.response_time_ms < 1500, message=f"TTFB latency: {p.response_time_ms:.0f}ms"))

df_findings = pd.DataFrame([f.model_dump() for f in findings])
display(df_findings[["check_id", "category", "severity", "passed", "message"]])

In [ ]:
# Cell 8: Evidence-Based Scoring Calculation
total_checks = len(findings)
passed_checks = sum(1 for f in findings if f.passed)
critical_fails = sum(1 for f in findings if not f.passed and f.severity == "critical")

raw_score = (passed_checks / total_checks) * 100.0 if total_checks else 100.0

# Apply critical failure caps
final_score = raw_score
if critical_fails >= 2:
    final_score = min(final_score, 45.0)
elif critical_fails == 1:
    final_score = min(final_score, 65.0)

grade = "A" if final_score >= 90 else ("B" if final_score >= 80 else ("C" if final_score >= 70 else "D"))

print(f"📊 Audit Result: {final_score:.1f} / 100 (Grade {grade})")
print(f"Passed Checks: {passed_checks}/{total_checks} | Critical Issues: {critical_fails}")

In [ ]:
# Cell 9: Interactive Plotly Gauge Visualization
fig = go.Figure(go.Indicator(
    mode="gauge+number",
    value=final_score,
    title={"text": f"SEO Health Score: {TARGET_URL}", "font": {"size": 22}},
    gauge={
        "axis": {"range": [0, 100]},
        "bar": {"color": "#00f0ff"},
        "steps": [
            {"range": [0, 40], "color": "#ef4444"},
            {"range": [40, 70], "color": "#f59e0b"},
            {"range": [70, 90], "color": "#3b82f6"},
            {"range": [90, 100], "color": "#10b981"}
        ],
        "threshold": {
            "line": {"color": "white", "width": 4},
            "thickness": 0.75,
            "value": final_score
        }
    }
))
fig.update_layout(paper_bgcolor="#0b0f19", font={"color": "white"}, width=600, height=350)
fig.show()

In [ ]:
# Cell 10: Grounded Action Plan Synthesis (Deterministic Fallback)
action_items = []
rank = 1
for f in findings:
    if not f.passed:
        action_items.append({
            "rank": rank,
            "check_id": f.check_id,
            "action": f"Resolve {f.check_id}: {f.message}",
            "severity": f.severity.upper(),
            "impact": "HIGH" if f.severity == "critical" else "MEDIUM",
            "effort": "LOW"
        })
        rank += 1

print("🎯 Prioritized Action Plan:")
for it in action_items:
    print(f"  #{it['rank']} [{it['severity']}] {it['action']} (Impact: {it['impact']}, Effort: {it['effort']})")

In [ ]:
# Cell 11: Export Single-File HTML Report
html_out = f"""<!DOCTYPE html>
<html>
<head><title>Audit Report - {TARGET_URL}</title></head>
<body style='font-family: sans-serif; background: #0b0f19; color: white; padding: 30px;'>
  <h1 style='color: #00f0ff;'>AuditPilot SEO Report</h1>
  <p>Target: <strong>{TARGET_URL}</strong> | Score: <strong>{final_score:.1f}/100</strong> (Grade {grade})</p>
  <h3>Action Items:</h3>
  <ul>{''.join(f'<li>{it["action"]}</li>' for it in action_items)}</ul>
</body></html>"""

with open("colab_audit_report.html", "w", encoding="utf-8") as f:
    f.write(html_out)

print("💾 Report successfully written to colab_audit_report.html")